# 0️⃣4️⃣ - How to Execute Commands on Devices

![Persona](https://img.shields.io/badge/%F0%9F%91%A4%20Persona-%F0%9F%A4%96%20Network%20Automation%20Developer-lightgrey) ![Difficulty](https://img.shields.io/badge/%F0%9F%9A%A6%20Difficulty-Beginner-green) ![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.13%2B-yellow?logo=python&logoColor=white)

> In this playbook we explore how to inspect your RADKit inventory, target the right devices, and execute CLI commands on one or many devices.

---

### 📋 What you'll learn

| # | Topic |
|---|---|
| 1 | How the `Device` and `DeviceDict` objects expose your inventory |
| 2 | How to filter devices before running commands |
| 3 | How to execute commands on a single device and validate the result |
| 4 | How to run one or many commands across multiple devices |
| 5 | How to handle partial failures and choose the right execution pattern |

### 🛠️ Before You Begin

If you haven't set up your environment yet, follow the instructions in [SETUP.md](../SETUP.md) before running any cells.

> **Prerequisite:** Complete [playbook 02](02-connect-to-my-service.ipynb) first, as this notebook assumes you can authenticate and obtain a `Service` object.

---

### 🤖 The `Device` and `DeviceDict` Objects

`Service.inventory` gives you access to the devices registered in your RADKit service.

- A `Device` represents one managed endpoint that you can inspect and execute commands on
- A `DeviceDict` is a filterable collection of `Device` objects that lets you target many devices at once

In practice, most workflows in this notebook follow the same pattern: start from `service.inventory`, narrow the target set, and then call `exec()` on either one `Device` or a `DeviceDict`.

---

## 📚 Method 1: Inspect and Filter Your Inventory

**Best for:** Understanding what is available in your service before you target devices with automation.

**How it works:**
1. Read devices from `service.inventory`.
2. Inspect useful attributes such as name, type, and host.
3. Filter the inventory down to the exact devices you want.

**What you need:**
- Your CCO user ID
- Your RADKit Service ID

---

### 1.1 List Inventory Details

This first example walks the full inventory and prints a few attributes for every onboarded device.

In [2]:
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()

    print("")
    # Going through all the devices in the inventory and printing their parameters
    for device in service.inventory.values():
        print(f"📱 Device {device.name} is type {device.device_type} and has host {device.host} ...")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.

📱 Device n540-9 is type IOS_XR and has host 10.62.154.211 ...
📱 Device dnac is type CATALYST_CENTER and has host 10.103.23.1 ...
📱 Device aci-erik-simulator-admin is type APIC and has host 10.62.188.211 ...
📱 Device ksp-g03-n540-22 is type IOS_XR and has host 10.62.189.43 ...
📱 Device n540-10 is type IOS_XR and has host 10.62.154.212 ...
📱 Device ksp-g03-n540-23 is type IOS_XR and has host 10.62.189.44 ...
📱 Device p0-2e is type IOS_XE and has host 10.48.172.59 ...
📱 Device ccenter is type CATALYST_CENTER and has host 10.10.41.1 ...
📱 Device nexus-dashboard is type NEXUS_DASHBOARD and has host 10.48.50.141 ...
📱 Device cc-sagradas is type CATALYST_CENTER and has host 172.28.60.227 ...
📱 Device ksp-g03-c2960s-16 is type IOS_XE and has host 10.62.189.27 ...
📱 Device ksp-g02-asr9010-02 is type IOS_XR and has host 10.62.189.30 ...
📱 Device ksp-g03-n54

---

### 1.2 Filter by Device Type

Use `inventory.filter()` when you only want devices that match a given attribute. In this example, the notebook narrows the inventory to `IOS_XR` devices before printing their names.

In [3]:
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()

    print("")
    # Filtering only the IOS-XR devices
    for device in service.inventory.filter("device_type", "IOS_XR").values():
        print(f"📱 Device {device.name} is type {device.device_type} ...")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.

📱 Device n540-9 is type IOS_XR ...
📱 Device ksp-g03-n540-22 is type IOS_XR ...
📱 Device n540-10 is type IOS_XR ...
📱 Device ksp-g03-n540-23 is type IOS_XR ...
📱 Device ksp-g02-asr9010-02 is type IOS_XR ...
📱 Device ksp-g03-n540-19 is type IOS_XR ...
📱 Device ksp-g02-asr9010-01 is type IOS_XR ...
📱 Device ksp-g04-asr9912-04 is type IOS_XR ...


---

### 1.3 Combine Multiple Filters

You can combine filtered inventories with `|` to build a broader target set. Here we merge `IOS_XR` and `IOS_XE` devices into one `DeviceDict`.

In [4]:
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()

    # Combining two filtered inventories with | (union) to get both IOS_XR and IOS_XE devices
    print("")
    ios_devices = service.inventory.filter("device_type", "IOS_XR") | service.inventory.filter("device_type", "IOS_XE")
    for device in ios_devices.values():
        print(f"📱 Device {device.name} is type {device.device_type} ...")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.

📱 Device n540-9 is type IOS_XR ...
📱 Device ksp-g03-n540-22 is type IOS_XR ...
📱 Device n540-10 is type IOS_XR ...
📱 Device p0-2e is type IOS_XE ...
📱 Device ksp-g03-n540-23 is type IOS_XR ...
📱 Device ksp-g03-c2960s-16 is type IOS_XE ...
📱 Device ksp-g02-asr9010-02 is type IOS_XR ...
📱 Device ksp-g03-n540-19 is type IOS_XR ...
📱 Device ksp-g02-asr9010-01 is type IOS_XR ...
📱 Device ksp-g01-asr1001x-10 is type IOS_XE ...
📱 Device ksp-g04-asr9912-04 is type IOS_XR ...


---

## ⚙️ Method 2: Execute Commands on a Single Device

**Best for:** Targeted reads or changes when you already know which device you want to work with.

**How it works:**
1. Select one `Device` from `service.inventory`.
2. Call `exec()` with one command string or a configuration payload.
3. Use `wait()` to retrieve the final `SingleExecResponse`.
4. Check `status`, then inspect `data`, `raw_data`, or `errors`.

**What you need:**
- A reachable device name from your inventory
- The CLI command or configuration payload you want to send

---

### 2.1 Retrieve Operational Output

This example runs `show ip interface brief` on one IOS XR device and inspects the fields returned by `SingleExecResponse`.

> 💡 `wait()` also accepts a timeout such as `wait(5)` if you want the request to fail fast.

In [5]:
from radkit_client import Client
from radkit_client import ExecStatus

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()

    my_device = service.inventory['ksp-g02-asr9010-02']
    exec_result = my_device.exec("show version").wait()
    
    if exec_result.status == ExecStatus.SUCCESS:
        print(f"\n✅ Execution Status: {exec_result.status}")
        print(f"📟 Device name: {exec_result.device_name}")
        print(f"🧬 Device type: {exec_result.device_type}")
        print(f"🧾 Device command: {exec_result.command}")
        print(f"🆔 Client ID: {exec_result.client_id}")
        print(f"☁️ Service ID: {exec_result.service_id}")
        print(f"📦 Raw Data: {exec_result.raw_data}")
    else:
        print(f"\n❌ Command execution failed: {str(exec_result.errors)}")
    


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.

✅ Execution Status: ExecStatus.SUCCESS
📟 Device name: ksp-g02-asr9010-02
🧬 Device type: IOS_XR
🧾 Device command: show version
🆔 Client ID: alfsando@cisco.com
☁️ Service ID: uhfm-4hm2-b5bx
📦 Raw Data: RP/0/RSP0/CPU0:KSP-G02-ASR9010-02#show version
Wed Apr  1 12:13:04.794 GMT+4
Cisco IOS XR Software, Version 7.10.2
Copyright (c) 2013-2023 by Cisco Systems, Inc.
 
Build Information:
 Built By     : deenayak
 Built On     : Sat Nov 18 06:46:57 PST 2023
 Built Host   : iox-ucs-035
 Workspace    : /auto/srcarchive14/prod/7.10.2/asr9k-x64/ws
 Version      : 7.10.2
 Location     : /opt/cisco/XR/packages/
 Label        : 7.10.2
 
cisco ASR9K () processor
System uptime is 17 weeks 1 day 27 minutes
 
RP/0/RSP0/CPU0:KSP-G02-ASR9010-02#


---

### 2.2 Diagnose Failed Single-Device Executions

If the target device is unreachable or the execution fails for another reason, the response still carries useful error details. This example repeats the same command against an unreachable device.

In [6]:
from radkit_client import Client
from radkit_client import ExecStatus

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()

    my_device = service.inventory['ksp-g03-n540-22']
    exec_result = my_device.exec("show ip interface brief").wait()
    
    if exec_result.status == ExecStatus.SUCCESS:
        print(f"\n✅ Execution Status: {exec_result.status}")
        print(f"📟 Device name: {exec_result.device_name}")
        print(f"🧬 Device type: {exec_result.device_type}")
        print(f"🧾 Device command: {exec_result.command}")
        print(f"🆔 Client ID: {exec_result.client_id}")
        print(f"☁️ Service ID: {exec_result.service_id}")
        print(f"📦 Raw Data: {exec_result.raw_data}")
    else:
        print(f"\n❌ Command execution failed: {str(exec_result.errors)}")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.

❌ Command execution failed: ['Device action failed: Timeout exception while preparing connection']


---

### 2.3 Push Configuration Commands

`exec()` is not limited to show commands. You can also send configuration-mode command strings and inspect the returned status just as you would for a read operation.

The following example creates a new loopback interface.

In [7]:
from radkit_client import Client
from radkit_client import ExecStatus

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()

    my_device = service.inventory['ksp-g02-asr9010-02']
    exec_result = my_device.exec("configure terminal\ninterface Loopback500\ndescription TestRADKit").wait()
    
    if exec_result.status == ExecStatus.SUCCESS:
        print(f"\n✅ Execution Status: {exec_result.status}")
        print(f"📦 Raw Data: {exec_result.raw_data}")
    else:
        print(f"❌ Command execution failed: {str(exec_result.errors)}")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.

✅ Execution Status: ExecStatus.SUCCESS
📦 Raw Data: RP/0/RSP0/CPU0:KSP-G02-ASR9010-02#configure terminal
 
Wed Apr  1 12:14:19.732 GMT+4
RP/0/RSP0/CPU0:KSP-G02-ASR9010-02(config)#interface Loopback500
RP/0/RSP0/CPU0:KSP-G02-ASR9010-02(config-if)#description TestRADKit
RP/0/RSP0/CPU0:KSP-G02-ASR9010-02(config-if)#


---

### 2.4 Validate CLI Output Carefully

> RADKit focuses on transport and execution delivery. If the device accepts the session but rejects the CLI syntax, the SDK may still mark the request as delivered.

Check the returned `data` or `raw_data` for device-side CLI errors. For deeper validation and structured parsing, pair this workflow with [playbook 05](05-parse-outputs-with-genie.ipynb).

---

## ⚙️⚙️⚙️ Method 3: Execute Across Multiple Devices

**Best for:** Fan-out operations where the same automation step must run against several devices or several commands at once.

**How it works:**
1. Build a `DeviceDict` from filters or manual additions.
2. Call `exec()` with one command or a list of commands.
3. Read results by device, by command, or by status depending on the response type.
4. Use `join()` when you want to launch requests independently and wait on them together.

**What you need:**
- A group of device names or a filter expression
- One command or a list of commands

The following sections show the main execution patterns.

---

### 3.1 Single Device, Multiple Commands

Pass a list of command strings to `exec()` when one device needs several CLI queries in a single request. The result is an instance of the `ExecResponse_ByCommand_ToSingle` class where the commands themselves are the key.

In [11]:
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()

    my_device = service.inventory['p0-2e']
    exec_result_multiple = my_device.exec(["show version | include Version|uptime", "show cdp neighbors"]).wait()
    print(f"\n📦 Output of command `show version | include Version|uptime` is: \n\n{exec_result_multiple['show version | include Version|uptime'].data}\n---------------\n")
    print(f"📦 Output of command `show cdp neighbors` is: \n\n{exec_result_multiple['show cdp neighbors'].data}\n---------------\n")



A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.

📦 Output of command `show version | include Version|uptime` is: 

p0-2E#show version | include Version|uptime
Cisco IOS XE Software, Version 17.18.02
Cisco IOS Software [IOSXE], Catalyst L3 Switch Software (CAT9K_IOSXE), Version 17.18.2, RELEASE SOFTWARE (fc3)
licensed under the GNU General Public License ("GPL") Version 2.0.  The
software code licensed under GPL Version 2.0 is free software that comes
GPL code under the terms of GPL Version 2.0.  For more details, see the
BOOTLDR: System Bootstrap, Version 17.15.1r, RELEASE SOFTWARE (P)
p0-2E uptime is 4 weeks, 19 hours, 16 minutes
Switch Ports Model              SW Version        SW Image              Mode   
p0-2E#
---------------

📦 Output of command `show cdp neighbors` is: 

p0-2E#show cdp neighbors
Capability Codes: R - Router, T - Trans Bridge, B - Source Route Bridge
                  S -

---

### 3.2 Multiple Devices, Single Command

Apply `exec()` to a `DeviceDict` when the same command should run across many devices. The response is an instance of `ExecResponse_ByDevice_ToSingle`, keyed by device name.

In [12]:
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    # Let's pick our first device by filtering based on the name. This gives us a DeviceDict object
    my_devices = service.inventory.filter('name','p0-2e')
    
    # Next, we add our second device to the same variable with the .add() method
    # Notice that we can simply pass the name of the device! The method will look it up in the inventory by itself
    my_devices.add('ksp-g03-c2960s-16')
    
    # We then execute the same command on both devices at the same time and store the results in a variable
    response = my_devices.exec("show version | include Version|uptime").wait()
    
    print("")
    for device_response in response.items():
        # The reuslting items are tuples where the first element is the device name, and the second is the SingleExecResponse object with all the details and data of the execution for that specific device
        print(f"📱 Device {device_response[0]} : (Status is {device_response[1].status}) Command '{device_response[1].command}' output is:\n{device_response[1].data}\n----------\n")
    
    


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.

📱 Device ksp-g03-c2960s-16 : (Status is ExecStatus.SUCCESS) Command 'show version | include Version|uptime' output is:
KSP-G03-C2960S-16#show version | include Version|uptime
Cisco IOS Software, C2960S Software (C2960S-UNIVERSALK9-M), Version 15.2(2)E9, RELEASE SOFTWARE (fc4)
BOOTLDR: C2960S Boot Loader (C2960S-HBOOT-M) Version 15.0(2r)SE, RELEASE SOFTWARE (fc1)
KSP-G03-C2960S-16 uptime is 20 weeks, 6 days, 4 hours, 0 minutes
Version ID                      : V01
Switch Ports Model                     SW Version            SW Image                 
KSP-G03-C2960S-16#
----------

📱 Device p0-2e : (Status is ExecStatus.SUCCESS) Command 'show version | include Version|uptime' output is:
p0-2E#show version | include Version|uptime
Cisco IOS XE Software, Version 17.18.02
Cisco IOS Software [IOSXE], Catalyst L3 Switch Software (CAT9K_IOSXE), Version 17.

---

### 3.3 Multiple Devices, Multiple Commands

The same `DeviceDict` pattern also supports a list of commands. In that case, the result becomes an instance of the class `ExecResponse_ByDevice_ByCommand`, which is a nested structure indexed by device and command, as shown below:

In [13]:
from radkit_client import Client

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    # Our two devices of interest in a DeviceDict object
    my_devices = service.inventory.filter('name', 'p0-2e')
    my_devices.add('ksp-g03-c2960s-16')
    
    commands = [
        "show version | include Version|uptime",
        "show processes cpu | include one minute",
    ]
    response = my_devices.exec(commands).wait()
    
    for device_name in ['p0-2e', 'ksp-g03-c2960s-16']:
        print(f"\n📱 {device_name}")
        for command in commands:
            result = response[device_name][command]
            print(f"🧾 {command}")
            print(f"Status: {result.status}")
            print(f"{result.data}\n----------")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.

📱 p0-2e
🧾 show version | include Version|uptime
Status: ExecStatus.SUCCESS
p0-2E#show version | include Version|uptime
Cisco IOS XE Software, Version 17.18.02
Cisco IOS Software [IOSXE], Catalyst L3 Switch Software (CAT9K_IOSXE), Version 17.18.2, RELEASE SOFTWARE (fc3)
licensed under the GNU General Public License ("GPL") Version 2.0.  The
software code licensed under GPL Version 2.0 is free software that comes
GPL code under the terms of GPL Version 2.0.  For more details, see the
BOOTLDR: System Bootstrap, Version 17.15.1r, RELEASE SOFTWARE (P)
p0-2E uptime is 4 weeks, 19 hours, 17 minutes
Switch Ports Model              SW Version        SW Image              Mode   
p0-2E#
----------
🧾 show processes cpu | include one minute
Status: ExecStatus.SUCCESS
p0-2E#show processes cpu | include one minute
CPU utilization for five seconds: 0%/0%; one mi

---

### 3.4 Join Independent Requests

If you start several requests separately, `join()` lets you wait for them together and display a progress bar while they complete.

The resulting variable is a `JoinedTasks` instance, with its own set of attributes, as seen below:

In [14]:
from radkit_client import Client
from radkit_client.sync.joining import join

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    # We execute a command on two devices independently
    # Notice that we are not using the wait() method here
    print("")
    command_request_1 = service.inventory['p0-2e'].exec("show version | include Version|uptime")
    command_request_2 = service.inventory['ksp-g03-c2960s-16'].exec("show version | include Version|uptime")
    
    # We now join the requests
    with join(command_request_1, command_request_2) as joined_request:
        # Display of a progress bar while waiting for the results
        joined_request.show_progress()
        
        # Single wait for all the underlying requests to be completed
        joined_request.wait()
        
        # Status information of the requests within, once they are completed
        print(f"\n📋 Execution status: {joined_request.status}")
        print(f"✅ {joined_request.success_count} requests succeeded, ❌ {joined_request.failure_count} requests failed")
        
        # Individual results of each request within the joined request
        for request in joined_request.items():
            # Once done, we read the results. The resulting JoinedTasks object has individual tuples, where the first element is the index, and the second is the SingleExecResponse instance we will work with
            print(f"\n\n📱 Device {request[1].device.name} : (Status is {request[1].status}) Command '{request[1].command}' output is:\n\n{request[1].data}\n----------")



A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.



Tasks completed:    0.0% [>                             ]   0/  2 eta [?:??:??]






                    5      ===============>                ]   1/  2 eta [00 01Tasks completed:  100.0% [===============================>]   2/  2 eta [00:00]

📋 Execution status: JoinedTasksStatus.SUCCESS
✅ 2 requests succeeded, ❌ 0 requests failed


📱 Device p0-2e : (Status is ExecStatus.SUCCESS) Command 'show version | include Version|uptime' output is:

p0-2E#show version | include Version|uptime
Cisco IOS XE Software, Version 17.18.02
Cisco IOS Software [IOSXE], Catalyst L3 Switch Software (CAT9K_IOSXE), Version 17.18.2, RELEASE SOFTWARE (fc3)
licensed under the GNU General Public License ("GPL") Version 2.0.  The
software code licensed under GPL Version 2.0 is free software that comes
GPL code under the terms of GPL Version 2.0.  For more details, see the
BOOTLDR: System Bootstrap, Version 17.15.1r, RELEASE SOFTWARE (P)
p0-2E uptime is 4 weeks, 19 hours, 18 minutes
Switch Ports Model           

---

### 3.4.1 Handle Completed Requests as They Come

It is possible to yield completed requests as soon as they finish. For this purpose, use the `as_completed()` method from the `JoinedTasks` instance.

Each yielded item is a `SingleExecResponse`, as shown below:

In [15]:
from radkit_client import Client
from radkit_client.sync.joining import join
from datetime import datetime

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    # We execute a command on two devices independently
    # Notice that we are not using the wait() method here
    print("")
    command_request_1 = service.inventory['p0-2e'].exec("show version | include Version|uptime")
    command_request_2 = service.inventory['ksp-g03-c2960s-16'].exec("show version | include Version|uptime")
    
    # We now join the requests
    with join(command_request_1, command_request_2) as joined_request:
        # Yielding of completed requests as they complete, without waiting for all of them to be completed
        for request_done in joined_request.as_completed():
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
            print(f"\n[⏱️ {timestamp}] 📱 Device {request_done.device.name} : (Status is {request_done.status}) Command '{request_done.command}' output is:\n\n{request_done.data}\n----------")


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.


[⏱️ 2026-04-01 12:24:28.384] 📱 Device p0-2e : (Status is ExecStatus.SUCCESS) Command 'show version | include Version|uptime' output is:

p0-2E#show version | include Version|uptime
Cisco IOS XE Software, Version 17.18.02
Cisco IOS Software [IOSXE], Catalyst L3 Switch Software (CAT9K_IOSXE), Version 17.18.2, RELEASE SOFTWARE (fc3)
licensed under the GNU General Public License ("GPL") Version 2.0.  The
software code licensed under GPL Version 2.0 is free software that comes
GPL code under the terms of GPL Version 2.0.  For more details, see the
BOOTLDR: System Bootstrap, Version 17.15.1r, RELEASE SOFTWARE (P)
p0-2E uptime is 4 weeks, 19 hours, 18 minutes
Switch Ports Model              SW Version        SW Image              Mode   
p0-2E#
----------

[⏱️ 2026-04-01 12:24:28.674] 📱 Device ksp-g03-c2960s-16 : (Status is ExecStatus.SUCCESS) Command '

---

### 3.5 Handle Partial Failures

When one device succeeds and another fails, RADKit can still return a partially successful response. Use `by_status` to isolate the failed entries and inspect their errors. Each of the resulting items is an instance of the `SingleExecResponse` class:

In [16]:
from radkit_client import Client
from radkit_client import ExecStatus

user_id = input("👤 Enter your CCO user id: ")
service_id = input("🛠️ Enter the service id you want to connect to: ")

with Client.create() as client:
    client.sso_login(user_id)
    service = client.service_cloud(service_id).wait()
    
    # Let's pick our first device by filtering based on the name. This gives us a DeviceDict object
    my_devices = service.inventory.filter('name','p0-2e')
    
    # This device is offline
    my_devices.add('n540-10')
    
    # We then execute the same command on both devices at the same time and store the results in a variable
    response = my_devices.exec("show version").wait()
    
    # If the execution was partially successful, we extract the unsuccessful devices and print the error message for each of them
    if response.status == ExecStatus.PARTIAL_SUCCESS:
        failed_devices = response.by_status['FAILURE'].items()
        for index, failed_device in failed_devices:
            print(f"\n\n❌ Failed device: {failed_device.device_name}")
            print(f"Command: {failed_device.command}")
            print(f"Errors: {failed_device.errors}")
            print(f"Status: {failed_device.status}\n----------\n")        


A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.


❌ Failed device: n540-10
Command: show version
Errors: ['Device action failed: Timeout exception while preparing connection']
Status: ExecStatus.FAILURE
----------



---

## 🔀 Which Method Should I Use?

| | 📚 Inventory inspection | ⚙️ Single-device execution | ⚙️⚙️⚙️ Multi-device execution |
|---|---|---|---|
| **Primary object** | `DeviceDict` from `service.inventory` | `Device` | `DeviceDict` |
| **Best when** | You are discovering devices or building a target list | You need precise reads or changes on one device | You need to fan out commands across many targets |
| **Typical input** | Filters such as name or device type | One device name plus one command payload | A device group plus one or many commands |
| **Class types you get** | `Device` and `DeviceDict` | `SingleExecResponse` | `ExecResponse_ByCommand_ToSingle`, `ExecResponse_ByDevice_ToSingle`, `ExecResponse_ByDevice_ByCommand`, or `JoinedTasks` depending on the pattern |
| **Result shape** | Iterables of `Device` objects | One execution result for one target | Results keyed by command, by device, or yielded from joined tasks |
| **Strength** | Safe way to inspect before acting | Rich status and output for one target | High efficiency across many targets |
| **Watch out for** | Broad filters that match more than intended | CLI syntax errors can appear only in device output | Partial failures require status-based filtering |